In [124]:
import pandas as pd
import numpy as np
import scipy.sparse as sp
import surprise
from surprise import SVD

In [125]:

# Load datasets
movies = pd.read_csv(r'..\Data\raw\movies.csv')
ratings = pd.read_csv(r'..\Data\raw\ratings.csv')


movies.head()




,movieId,title,genres
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,2,Jumanji (1995),Adventure|Children|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance
4,5,Father of the Bride Part II (1995),Comedy


In [126]:
ratings.head()

,userId,movieId,rating,timestamp
0,1,1,4.0,964982703
1,1,3,4.0,964981247
2,1,6,4.0,964982224
3,1,47,5.0,964983815
4,1,50,5.0,964982931


In [127]:
# merge the two datasets 
movie_ratings = pd.merge(ratings, movies,on='movieId')
movie_ratings.head()

,userId,movieId,rating,timestamp,title,genres
0,1,1,4.0,964982703,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,1,3,4.0,964981247,Grumpier Old Men (1995),Comedy|Romance
2,1,6,4.0,964982224,Heat (1995),Action|Crime|Thriller
3,1,47,5.0,964983815,Seven (a.k.a. Se7en) (1995),Mystery|Thriller
4,1,50,5.0,964982931,"Usual Suspects, The (1995)",Crime|Mystery|Thriller


In [128]:
# Lets do a check to see if the data is sparse or not.

print("Rows:", movie_ratings.shape[0])
print("Columns:", movie_ratings.columns.tolist())
print("Lowest rating:", movie_ratings.rating.min())
print("Highest rating:", movie_ratings.rating.max())
print("Number of users:", movie_ratings.userId.nunique())
print("Number of movies:", movie_ratings.movieId.nunique())
print("Duplicate user-movie pairs:", movie_ratings.duplicated(['userId', 'movieId']).sum())

Rows: 100836
Columns: ['userId', 'movieId', 'rating', 'timestamp', 'title', 'genres']
Lowest rating: 0.5
Highest rating: 5.0
Number of users: 610
Number of movies: 9724
Duplicate user-movie pairs: 0


In [129]:
total_cells = movie_ratings.userId.nunique() * movie_ratings.movieId.nunique()
filled_cells = len(movie_ratings)
print(f"Filled: {filled_cells / total_cells:.2%}")
print(f"Empty:  {1 - filled_cells / total_cells:.2%}")

Filled: 1.70%
Empty:  98.30%


In [130]:
from sklearn.model_selection import train_test_split

train_df, test_df = train_test_split(movie_ratings, test_size=0.2, random_state=42)

print("Training ratings:", len(train_df))
print("Test ratings:", len(test_df))

Training ratings: 80668
Test ratings: 20168


In [131]:
print("Users in test but not in train:", len(set(test_df.userId) - set(train_df.userId)))
print("Movies in test but not in train:", len(set(test_df.movieId) - set(train_df.movieId)))

Users in test but not in train: 0
Movies in test but not in train: 741


In [132]:
unseen_movies = set(test_df.movieId) - set(train_df.movieId)
affected = test_df[test_df.movieId.isin(unseen_movies)]
print("Test ratings on unseen movies:", len(affected))
print(f"Share of the test set: {len(affected) / len(test_df):.2%}")


Test ratings on unseen movies: 813
Share of the test set: 4.03%


In [133]:
from surprise import Dataset, Reader, SVD,accuracy
reader = Reader(rating_scale=(0.5, 5.0))
train_data = Dataset.load_from_df(train_df[['userId', 'movieId', 'rating']], reader)
trainset = train_data.build_full_trainset()
print("Trainset built with", trainset.n_users, "users and", trainset.n_items, "items.")

Trainset built with 610 users and 8983 items.


In [134]:
# MODEL A
testset = list(test_df[['userId', 'movieId', 'rating']].itertuples(index=False, name=None))

svd = SVD(random_state=42)
svd.fit(trainset)

predictions = svd.test(testset)
accuracy.rmse(predictions)


RMSE: 0.8831


np.float64(0.8831049180857837)

In [135]:

from numpy import average
from surprise import BaselineOnly

#Lazy model 1: same guess for everyone (the overall average)

global_mean = train_df.rating.mean()
print("Overall average rating:", round(global_mean, 2))

errors = (test_df.rating - global_mean) ** 2
print("Lazy model RMSE:", round(np.sqrt(errors.mean()), 4))

#Lazy model 2: average, plus "this user rates high/low", plus "this movie is popular/unpopular"
from surprise import BaselineOnly
baseline = BaselineOnly(verbose=False)
baseline.fit(trainset)
print("Baseline RMSE:", round(accuracy.rmse(baseline.test(testset), verbose=False), 4))

Overall average rating: 3.5
Lazy model RMSE: 1.0488
Baseline RMSE: 0.8824


### the models baseline is soolid but lets try it b4 tuning/ improving it

### Lets answer Q1: “What should user 1 watch next?”

The logic:

1.List every movie the user has not rated yet.
2.Ask the model to guess the user’s rating for each one.
3.Sort the guesses from highest to lowest.
4.Show the top 10.

In [136]:
def recommend(model, user_id, n=10):
    # lookup of movieId -> title and genres, from your own movie_ratings table
    movie_info = (movie_ratings[['movieId', 'title', 'genres']]
                  .drop_duplicates('movieId')
                  .set_index('movieId'))

    # movies this user has already rated (we don't recommend those)
    already_rated = set(movie_ratings.loc[movie_ratings.userId == user_id, 'movieId'])

    # every movie they have NOT rated
    candidates = [m for m in movie_info.index if m not in already_rated]

    # ask the model to guess a rating for each one
    guesses = [(m, model.predict(user_id, m).est) for m in candidates]

    # best guess first, keep the top n
    guesses.sort(key=lambda pair: pair[1], reverse=True)
    top = guesses[:n]

    # attach titles so the output is readable
    result = movie_info.loc[[m for m, _ in top]].copy()
    result['predicted_rating'] = [round(r, 2) for _, r in top]
    return result.reset_index(drop=True)

In [137]:
print(recommend)

<function recommend at 0x000002B36682E560>


In [143]:
# Sanity check: does the model recommend movies that the user has already rated?
uid = 1
#uid=50
print("Movies user", uid, "rated 5 stars (a sample):")
loved = movie_ratings[(movie_ratings.userId == uid) & (movie_ratings.rating == 5)]
print(loved[['title', 'genres']].head(8).to_string(index=False))

print("\nRecommended for user", uid, ":")
print(recommend(svd, uid))


Movies user 1 rated 5 stars (a sample):
                                 title                         genres
           Seven (a.k.a. Se7en) (1995)               Mystery|Thriller
            Usual Suspects, The (1995)         Crime|Mystery|Thriller
                  Bottle Rocket (1996) Adventure|Comedy|Crime|Romance
                        Rob Roy (1995)       Action|Drama|Romance|War
                 Canadian Bacon (1995)                     Comedy|War
                      Desperado (1995)         Action|Romance|Western
                  Billy Madison (1995)                         Comedy
Dumb & Dumber (Dumb and Dumber) (1994)               Adventure|Comedy

Recommended for user 1 :
                                               title  \
0                   Shawshank Redemption, The (1994)   
1                                 Rear Window (1954)   
2                                  Casablanca (1942)   
3                                      Brazil (1985)   
4  Dr. Strangelove or: H

#### The sanity check showed it didnt recommend what the user has rated b4 , however it recommended movies that are high rated is it bcse th user has given high rates for the other movies b4?


### Q2.How personal is it? Lets test on random users.

In [144]:
lists = {}
for uid in [1, 50, 300]:
    lists[uid] = set(recommend(svd, uid)['title'])
    print("User", uid, ":", sorted(lists[uid]))
    print()

print("Shared by user 1 and 50:", len(lists[1] & lists[50]))
print("Shared by user 1 and 300:", len(lists[1] & lists[300]))

User 1 : ["Amelie (Fabuleux destin d'Amélie Poulain, Le) (2001)", 'Brazil (1985)', 'Casablanca (1942)', 'Dr. Strangelove or: How I Learned to Stop Worrying and Love the Bomb (1964)', 'Grand Day Out with Wallace and Gromit, A (1989)', 'Lawrence of Arabia (1962)', 'Rear Window (1954)', 'Shawshank Redemption, The (1994)', 'Spirited Away (Sen to Chihiro no kamikakushi) (2001)', 'The Machinist (2004)']

User 50 : ["Amelie (Fabuleux destin d'Amélie Poulain, Le) (2001)", 'Clerks (1994)', 'Departed, The (2006)', 'Hustler, The (1961)', 'Outlaw Josey Wales, The (1976)', 'Princess Bride, The (1987)', 'Rear Window (1954)', 'Reservoir Dogs (1992)', 'Thank You for Smoking (2006)', 'Unforgiven (1992)']

User 300 : ['Boondock Saints, The (2000)', 'Casablanca (1942)', 'Glory (1989)', 'Grand Day Out with Wallace and Gromit, A (1989)', "King's Speech, The (2010)", 'Lawrence of Arabia (1962)', 'Philadelphia Story, The (1940)', 'Princess Bride, The (1987)', 'Star Wars: Episode VI - Return of the Jedi (1983

In [145]:
# cross-validation to get a more robust estimate of RMSE
from surprise.model_selection import cross_validate
from surprise import BaselineOnly

settings = {
    "Baseline (yardstick)": None,
    "SVD default": {},
    "SVD more practice (40 epochs)": {"n_epochs": 40},
    "SVD more practice + hold back more": {"n_epochs": 40, "reg_all": 0.05},
    "SVD fewer taste numbers (20)": {"n_factors": 20},
}

for name, params in settings.items():
    model = BaselineOnly(verbose=False) if params is None else SVD(random_state=42, **params)
    result = cross_validate(model, train_data, measures=['rmse'], cv=3, verbose=False)
    print(f"{name}: {result['test_rmse'].mean():.4f}")

Baseline (yardstick): 0.8794
SVD default: 0.8859
SVD more practice (40 epochs): 0.8931
SVD more practice + hold back more: 0.8769
SVD fewer taste numbers (20): 0.8788


In [146]:
from surprise.model_selection import GridSearchCV
import pandas as pd

param_grid = {
    "n_factors": [20, 50],
    "reg_all": [0.05, 0.1, 0.15],
    "n_epochs": [30, 40],
}

gs = GridSearchCV(SVD, param_grid, measures=["rmse"], cv=3, n_jobs=-1)
gs.fit(train_data)

print("Best score:", round(gs.best_score["rmse"], 4))
print("Best settings:", gs.best_params["rmse"])

results = pd.DataFrame(gs.cv_results)[["mean_test_rmse", "params"]]
print(results.sort_values("mean_test_rmse").head(5).to_string(index=False))

Best score: 0.8733
Best settings: {'n_factors': 50, 'reg_all': 0.1, 'n_epochs': 40}
 mean_test_rmse                                             params
       0.873262  {'n_factors': 50, 'reg_all': 0.1, 'n_epochs': 40}
       0.873800  {'n_factors': 20, 'reg_all': 0.1, 'n_epochs': 40}
       0.874313 {'n_factors': 50, 'reg_all': 0.15, 'n_epochs': 40}
       0.874566 {'n_factors': 50, 'reg_all': 0.05, 'n_epochs': 40}
       0.874756 {'n_factors': 20, 'reg_all': 0.15, 'n_epochs': 40}


In [147]:
best_svd = SVD(**gs.best_params["rmse"], random_state=42)
best_svd.fit(trainset)

best_predictions = best_svd.test(testset)
accuracy.rmse(best_predictions)


RMSE: 0.8729


np.float64(0.8728937225666182)

In [149]:
# to check the recommendations for user 1 with the best SVD model
print(recommend(best_svd, 1).to_string())

                                             title                    genres  predicted_rating
0                                       Ran (1985)                 Drama|War              5.00
1                 Streetcar Named Desire, A (1951)                     Drama              5.00
2                              Hustler, The (1961)                     Drama              5.00
3                     It Happened One Night (1934)            Comedy|Romance              5.00
4                            Paths of Glory (1957)                 Drama|War              5.00
5                           His Girl Friday (1940)            Comedy|Romance              4.99
6                             Touch of Evil (1958)  Crime|Film-Noir|Thriller              4.99
7                 Shawshank Redemption, The (1994)               Crime|Drama              4.99
8  Seventh Seal, The (Sjunde inseglet, Det) (1957)                     Drama              4.99
9                          Five Easy Pieces (1970)

In [ ]:
# do a count of how many ratings each movie has received, so we can see if the model is recommending popular movies or not
title_counts = movie_ratings.groupby('title').size()

rec = recommend(best_svd, 1)
rec['num_ratings'] = rec['title'].map(title_counts)
print(rec[['title', 'predicted_rating', 'num_ratings']].to_string(index=False))

lists = {u: set(recommend(best_svd, u)['title']) for u in [1, 50, 300]}
print("\nShared by user 1 and 50:", len(lists[1] & lists[50]))
print("Shared by user 1 and 300:", len(lists[1] & lists[300]))


                                          title  predicted_rating  num_ratings
                                     Ran (1985)              5.00           15
               Streetcar Named Desire, A (1951)              5.00           20
                            Hustler, The (1961)              5.00           18
                   It Happened One Night (1934)              5.00           14
                          Paths of Glory (1957)              5.00           12
                         His Girl Friday (1940)              4.99           14
                           Touch of Evil (1958)              4.99           17
               Shawshank Redemption, The (1994)              4.99          317
Seventh Seal, The (Sjunde inseglet, Det) (1957)              4.99           18
                        Five Easy Pieces (1970)              4.98            8

Shared by user 1 and 50: 7
Shared by user 1 and 300: 6


In [151]:
# Save the model


import os
import joblib

os.makedirs("../models", exist_ok=True)   # your notebook is in Notebooks/, so ../models is the project's models folder

# 1. retrain with the winning settings on ALL ratings
full_data = Dataset.load_from_df(movie_ratings[['userId', 'movieId', 'rating']], reader)
full_trainset = full_data.build_full_trainset()

final_svd = SVD(n_factors=50, reg_all=0.1, n_epochs=40, random_state=42)
final_svd.fit(full_trainset)

# 2. save the model
joblib.dump(final_svd, "../models/svd.pkl")

# 3. save the two small tables the app will need
movie_ratings[['movieId', 'title', 'genres']].drop_duplicates('movieId').to_csv("../models/movies.csv", index=False)
movie_ratings[['userId', 'movieId']].to_csv("../models/seen.csv", index=False)

# 4. check the files exist and how big they are
for f in ["svd.pkl", "movies.csv", "seen.csv"]:
    print(f, round(os.path.getsize(f"../models/{f}") / 1e6, 2), "MB")

svd.pkl 7.12 MB
movies.csv 0.49 MB
seen.csv 0.99 MB
